# Missing Value Audit and Cleaning

This notebook checks the cleaned interim dataset for missing values, applies the treatment decisions, validates the result, and saves a cleaned version for downstream analysis.

## Task checklist

- [x] 1. Load the interim dataset (cell 2)
- [x] 2. Audit missing values (cell 2)
- [x] 3. Remove unused columns (cell 3)
- [x] 4. Handle missing manufacturer values and save the cleaned dataset (cell 4)
- [x] 5. Validate the cleaned dataset and saved file (cell 5)
- [x] 6. Confirm the output is ready for downstream analysis (cell 5)


## Tasks 1-2: Load and audit the data

Load the interim dataset and inspect its shape and missing-value counts. This establishes the starting point for the cleaning decisions.


In [10]:
import pandas as pd

path = "../data/interim/cleaned_empty_columns_products_pricing_data.csv"
df = pd.read_csv(path)

print("Original shape:", df.shape)
print(df.isna().sum().sort_values(ascending=False).to_string())



Original shape: (7249, 26)
ean                    5706
manufacturer           4014
prices.shipping        2972
prices.amountMin          0
prices.amountMax          0
prices.availability       0
prices.currency           0
prices.dateSeen           0
prices.condition          0
id                        0
prices.merchant           0
prices.isSale             0
brand                     0
prices.sourceURLs         0
categories                0
dateAdded                 0
dateUpdated               0
asins                     0
imageURLs                 0
keys                      0
manufacturerNumber        0
name                      0
primaryCategories         0
sourceURLs                0
upc                       0
weight                    0


## Task 3: Decide and apply the treatment strategy

Remove low-value identifier and shipping columns that are not needed for downstream analysis. Keep the remaining fields unchanged for the next cleaning step.


In [11]:
# Remove low-value identifier and shipping columns that are not needed for analysis.
# These fields have a large number of missing values and are either redundant or not useful for the downstream KPI and modelling work.
df = df.drop(columns=['ean', 'prices.shipping', 'shipping'], errors='ignore')
print('Columns after removing ean and shipping:', df.columns.tolist())
print('Shape after removal:', df.shape)

Columns after removing ean and shipping: ['id', 'prices.amountMax', 'prices.amountMin', 'prices.availability', 'prices.condition', 'prices.currency', 'prices.dateSeen', 'prices.isSale', 'prices.merchant', 'prices.sourceURLs', 'asins', 'brand', 'categories', 'dateAdded', 'dateUpdated', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'name', 'primaryCategories', 'sourceURLs', 'upc', 'weight']
Shape after removal: (7249, 24)


## Task 4: Handle manufacturer values and save the cleaned dataset

Replace missing manufacturer names with `manufacturerNumber`, then save the cleaned dataframe while preserving the original interim dataset.


In [12]:
# Replace missing manufacturer names with the manufacturer number when available.
df["manufacturer"] = df["manufacturer"].replace(r"^\s*$", pd.NA, regex=True)
df["manufacturer"] = df["manufacturer"].fillna(df["manufacturerNumber"])
print("Manufacturer missing after fill:", df["manufacturer"].isna().sum())

# Save the cleaned dataset to the standard output path.
out_path = "../data/interim/cleaned_products_pricing_data.csv"
df.to_csv(out_path, index=False)
print(f"Saved cleaned dataset to: {out_path}")


Manufacturer missing after fill: 0
Saved cleaned dataset to: ../data/interim/cleaned_products_pricing_data.csv


## Tasks 5-6: Validate the result and confirm the output

Check that no missing values remain, the unused columns are absent, and the saved CSV matches the cleaned dataframe in shape and columns.


In [13]:
remaining_missing = df.isna().sum()
removed_columns = {"ean", "prices.shipping", "shipping"}
saved_df = pd.read_csv(out_path)

assert remaining_missing.sum() == 0, "Missing values remain in the cleaned dataset."
assert not removed_columns.intersection(df.columns), "Unused columns were not removed."
assert saved_df.shape == df.shape, "Saved dataset shape does not match the dataframe."
assert list(saved_df.columns) == list(df.columns), "Saved dataset columns do not match the dataframe."

print("Validation passed")
print("Final shape:", df.shape)
print("Saved file:", out_path)


Validation passed
Final shape: (7249, 24)
Saved file: ../data/interim/cleaned_products_pricing_data.csv
